# Capítulo 6 — RAG e agentes

**Livro:** começa na página **470** do PDF.

📗 **Gabarito**: soluções dos exercícios do caderno de prática. Tente sozinho antes de olhar. Cada chamada ao modelo custa centavos de centavo.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # onde ficam ia.py e os exercícios
from ia import *               # cliente, perguntar(), conversar(), embeddings(), gasto() ...
print("Pronto. Modelo padrão:", MODELO)

### Arquitetura RAG · PDF p. 475

#### 🧪 Exercício: Uma mini base de conhecimento

Crie uma lista `DOCS` com 6 textos curtos sobre uma empresa fictícia (horário, política de troca, frete, garantia, formas de pagamento, contato). Ela será usada nos próximos exercícios.

In [ ]:
DOCS = [
    "Horário de atendimento: segunda a sexta, das 8h às 18h; sábados, das 8h às 12h.",
    "Política de troca: produtos podem ser trocados em até 30 dias com a nota fiscal.",
    "Frete: grátis para compras acima de R$ 200 nas capitais; demais regiões pagam tabela dos Correios.",
    "Garantia: todos os equipamentos têm 12 meses de garantia contra defeitos de fabricação.",
    "Pagamento: aceitamos Pix, boleto e cartão em até 6 vezes sem juros.",
    "Contato: suporte@farmaciavida.com.br ou WhatsApp (11) 4000-0000.",
]
len(DOCS)

#### 🧪 Exercício: RAG completo: recuperar e responder

Escreva `responder(pergunta)`: recupera os 2 documentos mais parecidos (embeddings), coloca-os no prompt e pede ao modelo para responder **só com base neles**, dizendo "não sei" quando não estiver lá. Teste com uma pergunta que está e outra que não está na base.

In [ ]:
import numpy as np
v_docs = embeddings(DOCS)

def responder(pergunta, k=2):
    notas = similaridade(embeddings(pergunta), v_docs)[0]
    contexto = "\n".join(f"- {DOCS[i]}" for i in np.argsort(-notas)[:k])
    return perguntar(f"Contexto:\n{contexto}\n\nPergunta: {pergunta}",
                     sistema="Responda só com base no contexto. Se a resposta não estiver lá, diga 'Não sei'.",
                     temperature=0)

print(responder("O frete é grátis para Manaus?"))
print(responder("Vocês vendem perfumes importados?"))

### Algoritmos de Recuperação · PDF p. 477

#### 🧪 Exercício: Busca por palavras (BM25) × busca semântica

Para a pergunta *"Posso parcelar no cartão?"*, recupere o documento mais relevante de `DOCS` com **BM25** (`rank_bm25`) e com **embeddings**. Os dois acham o documento certo? Tente também *"Até quando posso devolver?"*.

In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np

bm25 = BM25Okapi([d.lower().split() for d in DOCS])
v_docs = embeddings(DOCS)

def buscar(pergunta):
    bm = DOCS[int(np.argmax(bm25.get_scores(pergunta.lower().split())))]
    sem = DOCS[int(np.argmax(similaridade(embeddings(pergunta), v_docs)[0]))]
    print(f"\n{pergunta}\n  BM25     : {bm}\n  Semântica: {sem}")

buscar("Posso parcelar no cartão?")
buscar("Até quando posso devolver?")

### Otimização de recuperação · PDF p. 498

#### 🧪 Exercício: Dividir textos longos (chunking)

Escreva `dividir(texto, tamanho=200, sobreposicao=50)` que corta um texto em pedaços de até `tamanho` caracteres, com `sobreposicao` caracteres repetidos entre um pedaço e o próximo. Teste com um parágrafo do livro.

In [ ]:
def dividir(texto, tamanho=200, sobreposicao=50):
    pedacos, inicio = [], 0
    while inicio < len(texto):
        pedacos.append(texto[inicio:inicio + tamanho])
        inicio += tamanho - sobreposicao
    return pedacos

texto = ("A recuperação funciona melhor quando os documentos são divididos em pedaços do tamanho certo. "
         "Pedaços muito grandes trazem ruído; pedaços muito pequenos perdem contexto. A sobreposição ajuda "
         "a não cortar uma ideia ao meio entre dois pedaços. ") * 3
for i, p in enumerate(dividir(texto)):
    print(i, len(p), repr(p[:60]) + "...")

### Ferramentas · PDF p. 515

#### 🧪 Exercício: Dar uma ferramenta ao modelo (function calling)

Crie a função `consultar_estoque(material)` (um dicionário fixo basta) e descreva-a como ferramenta (`tools=[...]`). Pergunte *"Quanto temos de dipirona?"*: o modelo deve pedir a ferramenta; execute-a e devolva o resultado para ele responder.

In [ ]:
import json
ESTOQUE = {"dipirona": 1200, "paracetamol": 0, "ibuprofeno": 350}

def consultar_estoque(material):
    return {"material": material, "quantidade": ESTOQUE.get(material.lower(), "não cadastrado")}

ferramentas = [{"type": "function", "function": {
    "name": "consultar_estoque", "description": "Quantidade em estoque de um material",
    "parameters": {"type": "object", "properties": {"material": {"type": "string"}}, "required": ["material"]}}}]

msgs = [{"role": "user", "content": "Quanto temos de dipirona?"}]
r = conversar(msgs, tools=ferramentas)
chamada = r.choices[0].message.tool_calls[0]
resultado = consultar_estoque(**json.loads(chamada.function.arguments))
msgs += [r.choices[0].message, {"role": "tool", "tool_call_id": chamada.id, "content": json.dumps(resultado)}]
print(conversar(msgs, tools=ferramentas).choices[0].message.content)

### Planejamento · PDF p. 521

#### 🧪 Exercício: Um agente simples com laço e limite de passos

Junte a ferramenta de estoque com uma de **calculadora** e escreva um laço: enquanto o modelo pedir ferramentas, execute-as e devolva os resultados; pare quando ele responder em texto ou depois de 5 passos. Pergunte: *"Se cada caixa tem 20 unidades, quantas caixas de ibuprofeno temos?"*

In [ ]:
import json
ESTOQUE = {"dipirona": 1200, "paracetamol": 0, "ibuprofeno": 350}
FUNCOES = {
    "consultar_estoque": lambda material: {"quantidade": ESTOQUE.get(material.lower(), 0)},
    "calcular": lambda expressao: {"resultado": eval(expressao, {"__builtins__": {}})},
}
ferramentas = [
    {"type": "function", "function": {"name": "consultar_estoque", "description": "Quantidade em estoque",
        "parameters": {"type": "object", "properties": {"material": {"type": "string"}}, "required": ["material"]}}},
    {"type": "function", "function": {"name": "calcular", "description": "Calcula uma expressão aritmética",
        "parameters": {"type": "object", "properties": {"expressao": {"type": "string"}}, "required": ["expressao"]}}},
]
msgs = [{"role": "user", "content": "Se cada caixa tem 20 unidades, quantas caixas de ibuprofeno temos?"}]
for passo in range(5):
    m = conversar(msgs, tools=ferramentas).choices[0].message
    if not m.tool_calls:
        print("Resposta final:", m.content); break
    msgs.append(m)
    for c in m.tool_calls:
        saida = FUNCOES[c.function.name](**json.loads(c.function.arguments))
        print(f"passo {passo}: {c.function.name}({c.function.arguments}) -> {saida}")
        msgs.append({"role": "tool", "tool_call_id": c.id, "content": json.dumps(saida)})

## Memória · PDF p. 552

#### 🧪 Exercício: Memória de conversa com resumo

Simule uma conversa de 6 mensagens. Quando o histórico passar de 4 mensagens, peça ao modelo um **resumo** das mais antigas e mantenha só o resumo + as 2 últimas. Pergunte no final algo que só o resumo lembra.

In [ ]:
historico = []

def falar(texto):
    global historico
    historico.append({"role": "user", "content": texto})
    if len(historico) > 4:
        antigas, recentes = historico[:-2], historico[-2:]
        resumo = perguntar("Resuma esta conversa em 2 frases, guardando nomes e números:\n"
                           + "\n".join(f"{m['role']}: {m['content']}" for m in antigas), temperature=0)
        historico = [{"role": "system", "content": "Resumo da conversa até aqui: " + resumo}] + recentes
    resposta = conversar(historico).choices[0].message.content
    historico.append({"role": "assistant", "content": resposta})
    return resposta

for frase in ["Meu nome é Rafael e cuido do MRP.", "Temos 1.200 caixas de dipirona.",
              "O lead time é de 30 dias.", "Fale de outra coisa: quem ganhou a Copa de 2002?"]:
    falar(frase)
print(falar("Qual é o meu nome e quantas caixas temos?"))

## 💰 Quanto gastei

In [ ]:
print(gasto())  # quanto esta sessão custou